# Eagle Eye: Isolation Forest experiment and evaluation

This notebook repeats the training pipeline of `ai/train.py` step by step so the team can see
and explain every stage. It uses only **synthetic** data from the project's simulator, so the
numbers show that the pipeline works on the simulated scenarios; they are **not** evidence of
real-world detection performance.

Run it from the project root with Jupyter (`pip install jupyterlab`, then `jupyter lab`).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "ai").exists() else Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from ai import dataset
from ai.evaluate import classification_metrics, detection_rate_by_scenario
from ai.features import FEATURE_NAMES, build_feature_table
from ai.model import normalize_score, raw_scores, train_model

## 1. Generate the synthetic dataset

40 synthetic employees, 90 calendar days (weekends skipped). The first 60 days contain only
normal behavior (training period). In the last 30 days about 10% of user-days are replaced by a
scenario day (B-H) so we have labels to evaluate against.

In [ ]:
events, labels = dataset.generate_dataset(seed=dataset.DEFAULT_SEED)
print(len(events), "raw events")
labels.groupby(["period", "scenario"]).size()

## 2. Feature engineering: one row per user per day

In [ ]:
features = build_feature_table(events)
table = features.merge(labels, on=["user_id", "day"])
table[FEATURE_NAMES].describe().T[["mean", "std", "min", "max"]].round(2)

## 3. Time-aware split and training

The model is trained only on the earlier, normal-only period and evaluated on the later period,
so it is never tested on the days it learned from.

In [ ]:
train = table[table.period == "train"]
test = table[table.period == "test"].copy()
model, calibration = train_model(train, seed=42)
calibration

## 4. From raw score to the 0-100 anomaly score

`raw = -model.score_samples(x)` (higher = more unusual).
`anomaly_score = 50 * (raw - typical) / (threshold - typical)`, clipped to 0..100, where *typical*
is the median and *threshold* the 99th percentile of raw scores on the training days.

In [ ]:
test["raw"] = raw_scores(model, test)
test["anomaly_score"] = [normalize_score(value, calibration) for value in test["raw"]]
test["predicted"] = (test["raw"] > calibration["threshold_raw_score"]).astype(int)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(test[test.label == 0]["anomaly_score"], bins=30, alpha=0.7, label="normal / benign days")
ax.hist(test[test.label == 1]["anomaly_score"], bins=30, alpha=0.7, label="suspicious scenario days")
ax.axvline(50, color="black", linestyle="--", label="anomaly threshold (50)")
ax.set_xlabel("anomaly score (0-100)")
ax.set_ylabel("user-days")
ax.legend()
plt.show()

## 5. Metrics on the test period (AI model alone)

In [ ]:
classification_metrics(test["label"], test["predicted"])

In [ ]:
pd.DataFrame(detection_rate_by_scenario(test)).T

## 6. What makes a day unusual?

Average feature values per scenario. Isolation Forest isolates days whose feature values are
rare compared with the training days.

In [ ]:
test.groupby("scenario")[["download_count", "download_mb", "delete_count", "night_login_count",
                          "usb_sensitive_transfer_count", "sensitive_access_count", "total_events"]].mean().round(1)

## 7. Findings and limitations

* The model alone catches the strongest, multi-signal days (combined anomaly) reliably, but
  misses many single-signal days such as one night login. This is why Eagle Eye combines it
  with explicit cybersecurity rules (`python -m scripts.evaluate_detection` evaluates the whole system).
* The benign "deadline day" scenario is often flagged: unusual is not the same as malicious.
  That is why alerts need an analyst decision.
* The data is synthetic and the scenarios were designed by us, so these numbers can be
  optimistic. Real behavior is noisier and real attacks are not labeled.